# SAE Alignment with Balanced Modes

This notebook demonstrates that a **Sparse Autoencoder (SAE)** trained on SSM state trajectories — with no access to A, B, C or the balancing transformation — independently recovers the balanced mode directions.

**Why this matters:** If the SAE finds the same directions as theory (via balanced realization), it confirms that the balanced modes are not a post-hoc mathematical artefact but reflect actual structure the model uses in its forward pass.

**Key result (from §3.6 of the paper):** Cosine similarity between the top SAE features and balanced modes ≥ 0.90 for well-separated tasks. For the Tiny Shakespeare NLP task, cosine sim reaches 0.963–0.994.

**SAE here is a validation tool, not a discovery tool.** For transformers, SAEs are used to find features (no canonical basis exists). For LTI SSMs, theory provides the balanced basis; the SAE verifies it.

**Reference:** §3.6 of the paper; `ssm_interpret.sae` module.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

import ssm_interpret as ssi

print(f'ssm_interpret {ssi.__version__}')

In [ ]:
# ── Task and model ─────────────────────────────────────────────────────────
T_SEQ, M, D_STATE = 32, 3, 3
RHOS = [0.99, 0.85, 0.50]
EPOCHS_SSM = 20000

def make_ema_data(n, T, rhos, seed=0):
    rng = np.random.default_rng(seed)
    u = rng.uniform(-1, 1, (n, T, len(rhos))).astype(np.float32)
    y = np.zeros_like(u)
    for k, rho in enumerate(rhos):
        s = 0.0
        for t in range(T):
            s = rho * s + (1 - rho) * u[:, t, k]
            y[:, t, k] = s
    return torch.tensor(u), torch.tensor(y)

class LinearSSM(nn.Module):
    """Standard linear SSM with full-A parameterization."""
    def __init__(self, m, d, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.A_raw = nn.Parameter(torch.randn(d, d) * 0.1)
        self.B = nn.Parameter(torch.randn(d, m) * 0.1)
        self.C = nn.Parameter(torch.randn(m, d) * 0.1)
        self.m, self.d = m, d

    def _A(self):
        sr = torch.linalg.matrix_norm(self.A_raw, ord=2)
        return self.A_raw / sr.clamp(min=1.0) * 0.99

    def forward(self, u):
        B_, T_, _ = u.shape
        A = self._A()
        x = torch.zeros(B_, self.d)
        ys = []
        for t in range(T_):
            x = x @ A.T + u[:, t, :] @ self.B.T
            ys.append(x @ self.C.T)
        return torch.stack(ys, dim=1)

    def forward_states(self, u):
        """Integration contract for ssi.collect_states()."""
        B_, T_, _ = u.shape
        A = self._A()
        x = torch.zeros(B_, self.d)
        xs, ys = [x.clone()], []
        for t in range(T_):
            x = x @ A.T + u[:, t, :] @ self.B.T
            xs.append(x.clone())
            ys.append(x @ self.C.T)
        return torch.stack(xs, dim=1), torch.stack(ys, dim=1)

    def get_matrices(self):
        with torch.no_grad():
            return (self._A().numpy().astype(np.float64),
                    self.B.detach().numpy().astype(np.float64),
                    self.C.detach().numpy().astype(np.float64))

u_tr, y_tr = make_ema_data(2048, T_SEQ, RHOS)
u_te, y_te = make_ema_data(512, T_SEQ, RHOS, seed=1)

print(f'Training SSM (d={D_STATE}, m={M})...')
model = LinearSSM(M, D_STATE, seed=0)
opt = optim.Adam(model.parameters(), lr=3e-3)
sched = optim.lr_scheduler.MultiStepLR(opt, milestones=[8000, 15000], gamma=0.3)
for ep in range(EPOCHS_SSM):
    opt.zero_grad()
    F.mse_loss(model(u_tr), y_tr).backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    sched.step()
    if (ep+1) % 5000 == 0:
        with torch.no_grad():
            print(f'  epoch {ep+1}: loss={F.mse_loss(model(u_tr), y_tr).item():.4e}')

A, B, C = model.get_matrices()
Ab, Bb, Cb, T_bal, sigma = ssi.balanced_realization(A, B, C, T=T_SEQ)
print(f'\nHSVs: {np.array(sigma).round(4)}')

## 1. Collect State Trajectories

In [ ]:
# Collect states using forward_states integration contract
X = ssi.collect_states(model, u_tr, method='explicit', max_seqs=20000)
print(f'State matrix X: {X.shape}  (N states × d dims)')
print(f'State mean: {X.mean(0).numpy().round(4)}')
print(f'State std:  {X.std(0).numpy().round(4)}')

## 2. Train SAE on State Trajectories

In [ ]:
D_HIDDEN = 2 * D_STATE  # overcomplete dictionary

print(f'Training SAE: d_input={D_STATE}, d_hidden={D_HIDDEN}')
print('Note: SAE has no access to A, B, C or T_bal — learns from state trajectories only')

sae = ssi.train_sae(
    X, d_hidden=D_HIDDEN,
    l1_weight=1e-3,
    epochs=3000,
    lr=1e-3,
    seed=0,
    verbose=True,
)

## 3. Alignment Score

In [ ]:
# Maximum cosine similarity between each balanced mode and any SAE feature
cos_sims = ssi.alignment_score(sae, T_bal, n_modes=D_STATE)
dead_frac = ssi.dead_fraction(sae, X, tol=0.05)

print('SAE alignment with balanced modes:')
for k, (cs, s) in enumerate(zip(cos_sims, np.array(sigma))):
    print(f'  mode k={k} (σ={s:.4f}): max|cos| = {cs:.4f}')

print(f'\nMean cosine similarity: {float(cos_sims.mean()):.4f}')
print(f'Min  cosine similarity: {float(cos_sims.min()):.4f}')
print(f'Dead SAE features:      {dead_frac:.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Alignment scores per mode
ax = axes[0]
sigma_np = np.array(sigma)
cos_np = cos_sims.numpy()
colors = ['#2196F3', '#4CAF50', '#FF9800'][:D_STATE]
ax.bar(range(D_STATE), cos_np, color=colors, edgecolor='k', linewidth=0.8)
ax.axhline(0.90, color='gray', linestyle='--', linewidth=1.5, label='0.90 threshold')
ax.set_xlabel('Balanced mode index k')
ax.set_ylabel('Max cosine similarity with SAE features')
ax.set_title(f'SAE Alignment (d_hidden={D_HIDDEN})')
ax.set_ylim(0, 1.05)
ax.legend()
ax.set_xticks(range(D_STATE))
for k, cs in enumerate(cos_np):
    ax.text(k, cs + 0.01, f'{cs:.3f}', ha='center', va='bottom', fontsize=10)

# Alignment vs HSV (correlation)
ax = axes[1]
ax.scatter(sigma_np, cos_np, c=colors, s=100, edgecolors='k', linewidth=0.8, zorder=3)
# Trend line
if D_STATE > 2:
    z = np.polyfit(sigma_np, cos_np, 1)
    x_line = np.linspace(sigma_np.min(), sigma_np.max(), 50)
    ax.plot(x_line, np.poly1d(z)(x_line), '--', color='gray', alpha=0.7)
    r = np.corrcoef(sigma_np, cos_np)[0, 1]
    ax.text(0.05, 0.95, f'r = {r:.2f}', transform=ax.transAxes,
            fontsize=11, va='top')
for k, (s, cs) in enumerate(zip(sigma_np, cos_np)):
    ax.annotate(f'k={k}', (s, cs), textcoords='offset points', xytext=(5, 5))
ax.set_xlabel('Hankel singular value σ_k')
ax.set_ylabel('SAE alignment (max|cos|)')
ax.set_title('Alignment Correlates with HSV')
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('sae_alignment.pdf', bbox_inches='tight')
plt.show()
print('Fig saved: sae_alignment.pdf')

## 4. Varying L1 Sparsity: Effect on Alignment

In [ ]:
l1_vals = [0.0, 1e-4, 1e-3, 5e-3, 1e-2]
mean_aligns = []
dead_fracs = []

print('Sweeping L1 sparsity weight...')
for l1 in l1_vals:
    sae_l1 = ssi.train_sae(X, d_hidden=D_HIDDEN, l1_weight=l1, epochs=2000, seed=0)
    cs = ssi.alignment_score(sae_l1, T_bal, n_modes=D_STATE)
    df = ssi.dead_fraction(sae_l1, X)
    mean_aligns.append(float(cs.mean()))
    dead_fracs.append(df)
    print(f'  l1={l1:.0e}: mean_align={float(cs.mean()):.4f}  dead={df:.1%}')

fig, ax1 = plt.subplots(figsize=(7, 4))
ax2 = ax1.twinx()
ax1.semilogx(l1_vals, mean_aligns, 'o-', color='#2196F3', label='Mean alignment', linewidth=2)
ax2.semilogx(l1_vals, dead_fracs, 's--', color='#F44336', label='Dead fraction', linewidth=2)
ax1.set_xlabel('L1 sparsity weight')
ax1.set_ylabel('Mean SAE alignment', color='#2196F3')
ax2.set_ylabel('Dead feature fraction', color='#F44336')
ax1.set_title('L1 Sparsity vs Alignment Tradeoff')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower left')
plt.tight_layout()
plt.savefig('sae_l1_sweep.pdf', bbox_inches='tight')
plt.show()
print('Fig saved: sae_l1_sweep.pdf')